# Monitoring and Alerting: Logistics SLA

This notebook provides continuous monitoring of logistics performance. According to the business requirements, we track the delay rate over time. If the percentage of delays (receiving goods after the committed date) exceeds the established threshold (15%), the script generates a system alert.

In a Databricks production environment, this code runs as a Job, and in case of an alert, it automatically sends a notification to the Procurement or Logistics team.

In [0]:
import os
import sys
from pathlib import Path

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import env
from src.monitoring.delay_alerts import run_delay_monitoring

print(f"Starting monitoring pipeline on {env.CATALOG}.{env.SILVER_SCHEMA}...")
monitoring_results = run_delay_monitoring(spark, threshold_pct=15.0)

metrics_df = monitoring_results["metrics"]
alerts_df = monitoring_results["alerts"]
print("Metrics calculated successfully.")

## 1. Delay Rate Over Time
This table shows the dynamics of delivery performance. 

In [0]:
display(metrics_df)

Databricks visualization. Run in Databricks to view.

## 2. Alert Trigger Verification
Here we filter only those periods where the delay rate exceeded the acceptable threshold of 15%. The following code cell checks for the presence of such records and, if any exist, artificially raises an error to stop the pipeline and trigger notifications.

In [0]:
display(alerts_df)

alert_count = alerts_df.count()

if alert_count > 0:
    problematic_months = [
        row["receipt_month"].strftime("%Y-%m") for row in alerts_df.collect()
    ]
    
    raise AssertionError(
        f"🚨 ALERT: Detected {alert_count} months with a delay rate > 15%! "
        f"Problematic periods: {', '.join(problematic_months)}. "
        "Logistics manager attention required."
    )
else:
    print("✅ All metrics are normal. The delay rate remains within the acceptable SLA (<= 15%).")